# Basic Question & Answer Generation from a FileSet

This notebook demonstrates the simplest way to generate question-answer pairs from a FileSet. Documents are chunked into seeds, then questions and labels are generated in a single step using `QuestionAndLabelGenerator`.

**Prerequisite**: Run `01_create_fileset.ipynb` first to create a FileSet and upload documents.

In [ ]:
%pip install lightningrod-ai python-dotenv -q

from IPython.display import clear_output
clear_output()

## Set up the client

Sign up at [dashboard.lightningrod.ai](https://dashboard.lightningrod.ai/?redirect=/api) to get your API key and **$50 of free credits**.

- **Google Colab**: Go to the Secrets section (key icon in left sidebar) and add a secret named `LIGHTNINGROD_API_KEY`
- **Local Jupyter**: Set the `LIGHTNINGROD_API_KEY` environment variable, or you'll be prompted to enter it

In [ ]:
from dotenv import load_dotenv
from lightningrod import LightningRod
from lightningrod.utils import config

load_dotenv()
api_key = config.get_config_value("LIGHTNINGROD_API_KEY")

lr = LightningRod(api_key=api_key)

## Configure FileSet ID

Paste the FileSet ID from notebook 1 below.

In [ ]:
fileset_id = "PASTE_YOUR_FILESET_ID_HERE"

## Configure the Question Pipeline

- **`FileSetSeedGenerator`** chunks the documents in your FileSet into seeds (text passages)
- **`QuestionAndLabelGenerator`** generates questions and answers in a single step — the simplest way to produce labeled Q&A pairs

In [ ]:
from lightningrod import (
    QuestionPipeline,
    FileSetSeedGenerator,
    QuestionAndLabelGenerator,
    BinaryAnswerType,
)

pipeline = QuestionPipeline(
    seed_generator=FileSetSeedGenerator(
        file_set_id=fileset_id,
        chunk_size=2000,
        chunk_overlap=200,
    ),
    question_generator=QuestionAndLabelGenerator(
        answer_type=BinaryAnswerType(),
        questions_per_seed=2,
        instructions=(
            "Generate binary yes/no questions about the financial metrics, business events, "
            "and forward guidance in these quarterly investor reports. Questions should be "
            "specific and verifiable from the report content."
        ),
    ),
)

## Run the Pipeline

In [ ]:
dataset = lr.transforms.run(
    pipeline,
    max_questions=10,
    name="FileSet - Basic QA",
)
print(f"Dataset: {dataset.id}")
print(f"Rows: {dataset.num_rows}")

> **Note:** This can take a few minutes to complete processing.

## View the Results

In [ ]:
%pip install pandas -q

from IPython.display import clear_output
clear_output()

In [ ]:
import pandas as pd

answer_type = BinaryAnswerType()
samples = dataset.download()
rows = dataset.flattened(answer_type)
df = pd.DataFrame(rows)

print(f"Generated {dataset.num_rows} samples ({dataset.valid_count() / dataset.num_rows * 100:.1f}% valid)\n")

cols = ["question_text", "label", "label_confidence", "is_valid"]
df[[c for c in cols if c in df.columns]]